# 02 — Lakehouse & Delta Lake Deep Dive

Every Fabric Lakehouse table is a **Delta table**. This notebook walks through the Delta features
you'll rely on in production: upserts, time travel, optimization, schema evolution, constraints
and Change Data Feed — using a banking "accounts/transactions" scenario throughout.


## 1. Create a Delta table and inspect it

In [ ]:
from pyspark.sql import functions as F

df_accounts = spark.createDataFrame([
    ("A100", "Rahul Mehta",  "SAVINGS",  15000.0, "ACTIVE"),
    ("A101", "Sana Iyer",    "CURRENT",   4200.0, "ACTIVE"),
    ("A102", "Wei Zhang",    "SAVINGS",   980.0,  "DORMANT"),
], ["account_id", "customer_name", "account_type", "balance", "status"])

df_accounts.write.format("delta").mode("overwrite").saveAsTable("accounts")

spark.sql("DESCRIBE DETAIL accounts").show(truncate=False)


## 2. Upserts with `MERGE INTO` (SCD Type 1)

This is the single most common Delta operation in real pipelines: applying a daily batch of
changed-data-capture (CDC) records onto a dimension/fact table.

In [ ]:
from delta.tables import DeltaTable

daily_updates = spark.createDataFrame([
    ("A101", "Sana Iyer",   "CURRENT", 5100.0, "ACTIVE"),   # balance changed
    ("A103", "Omar Farouk", "SAVINGS", 2200.0, "ACTIVE"),   # brand new account
], ["account_id", "customer_name", "account_type", "balance", "status"])

target = DeltaTable.forName(spark, "accounts")

(
    target.alias("t")
    .merge(daily_updates.alias("s"), "t.account_id = s.account_id")
    .whenMatchedUpdate(set={
        "balance": "s.balance",
        "status": "s.status",
        "customer_name": "s.customer_name",
    })
    .whenNotMatchedInsertAll()
    .execute()
)

spark.read.table("accounts").orderBy("account_id").show()


In [ ]:
%%sql
-- Same MERGE expressed in SQL, useful when orchestrated from a pipeline "Script" activity
MERGE INTO accounts AS t
USING daily_updates_view AS s
ON t.account_id = s.account_id
WHEN MATCHED THEN UPDATE SET t.balance = s.balance, t.status = s.status
WHEN NOT MATCHED THEN INSERT *


## 3. Time travel

Every write creates a new Delta version. You can query any prior version by number or timestamp —
invaluable for audits and debugging bad loads.

In [ ]:
spark.sql("DESCRIBE HISTORY accounts").select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

# Query as of a specific version
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("accounts")
df_v0.show()

# Query as of a timestamp
df_asof = spark.read.format("delta") \
    .option("timestampAsOf", "2026-01-01 00:00:00") \
    .table("accounts")


In [ ]:
%%sql
-- Roll back a table to a previous version (writes a NEW version that matches the old data)
RESTORE TABLE accounts TO VERSION AS OF 0


## 4. Table maintenance: `OPTIMIZE`, `ZORDER`, `VACUUM`

Small-file problems and stale files are the #1 cause of slow Fabric Spark jobs.

In [ ]:
%%sql
-- Compact small files into larger ones (recommended after heavy streaming/MERGE workloads)
OPTIMIZE accounts;

-- Co-locate rows by frequently-filtered columns for faster point/range lookups
OPTIMIZE accounts ZORDER BY (account_id);

-- Remove data files no longer referenced by the Delta log (default retention = 7 days)
VACUUM accounts RETAIN 168 HOURS;


Fabric also supports **V-Order** — a write-time optimization layered on top of Parquet that
speeds up reads from Power BI / Direct Lake and other Spark/SQL engines. It's on by default at the
workspace level, but can be set explicitly:

In [ ]:
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")

df_accounts.write.format("delta").mode("overwrite").saveAsTable("accounts_vordered")


## 5. Schema evolution

In [ ]:
df_accounts_v2 = df_accounts.withColumn("risk_score", F.lit(None).cast("double"))

(
    df_accounts_v2.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")   # allows the new risk_score column to be added
    .saveAsTable("accounts")
)

spark.read.table("accounts").printSchema()


## 6. Constraints & data quality guardrails

In [ ]:
%%sql
ALTER TABLE accounts ADD CONSTRAINT positive_balance CHECK (balance >= 0);
-- Any subsequent INSERT/MERGE/UPDATE violating this will fail fast instead of silently
-- corrupting the table.


## 7. Change Data Feed (CDF)

CDF lets *downstream* jobs read only the row-level changes since they last checked — the
foundation for efficient incremental Silver→Gold pipelines.

In [ ]:
%%sql
ALTER TABLE accounts SET TBLPROPERTIES (delta.enableChangeDataFeed = true);


In [ ]:
# Read only what changed between two versions
df_changes = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1)
    .table("accounts")
)

df_changes.select(
    "account_id", "balance", "_change_type", "_commit_version", "_commit_timestamp"
).show(truncate=False)


Next notebook: **03 — Data Ingestion & Transformation** covers reading messy real-world
sources and the transformation patterns (windows, pivots, UDFs) you'll layer on top of these Delta
fundamentals.